In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
plt.style.use("seaborn-v0_8")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

student_database = pd.read_csv('../data/student_job_dataset.csv')

encoder = LabelEncoder()
# For Degree
student_database['Degree'] = encoder.fit_transform(student_database['Degree'])
degree_mapping = dict(enumerate(encoder.classes_))
degree_reverse_mapping = {v: k for k, v in degree_mapping.items()}

encoder = LabelEncoder()
# For JobRole
student_database['JobRole'] = encoder.fit_transform(student_database['JobRole'])
JobRole_mapping = dict(enumerate(encoder.classes_))
JobRole_reverse_mapping = {v: k for k, v in JobRole_mapping.items()}

encoder = LabelEncoder()
# For Specialization
student_database['Specialization'] = encoder.fit_transform(student_database['Specialization'])
Specialization_mapping = dict(enumerate(encoder.classes_))
Specialization_reverse_mapping = {v: k for k, v in Specialization_mapping.items()}

X = student_database.drop(["JobRole"], axis=1)  # Features
y = student_database["JobRole"]                 # Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=200,        # number of trees
    max_depth=None,
    random_state=42,
    class_weight="balanced" # helpful if data is imbalanced
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

class JobPred:
    history = []

    def __init__(self, degree, specialization, cgpa, year_of_passing, certification):
        self.degree = degree
        self.specialization = specialization
        self.cgpa = cgpa
        self.year_of_passing = year_of_passing
        self.certification = certification

        self.case = {
            'input': {
                'degree': degree,
                'specialization': specialization,
                'cgpa': cgpa,
                'year_of_passing': year_of_passing,
                'certification': certification
            },
            'output': None
        }

    def predict(self):
        # IMPORTANT: order must match training data
        new_student = [[
            self.degree,
            self.specialization,
            self.cgpa,
            self.certification,
            self.year_of_passing
        ]]

        probabilities = rf_model.predict_proba(new_student)[0]
        top_3_indices = np.argsort(probabilities)[-3:][::-1]

        top_3_roles = [
            (JobRole_mapping[i], round(probabilities[i] * 100, 2))
            for i in top_3_indices
        ]

        self.case['output'] = top_3_roles
        JobPred.history.append(self.case)

        print("Top 3 Suitable Job Roles:")
        for role, confidence in top_3_roles:
            print(f"{role} → {confidence}%")

        # return top_3_roles

    @classmethod
    def get_history(cls):
        for i in cls.history:
            for role, confidence in i:
                print(f"{role} → {confidence}%")
        # return cls.history

Accuracy: 0.6171875
              precision    recall  f1-score   support

           0       0.45      0.51      0.48        65
           1       0.49      0.55      0.52       121
           2       0.24      0.18      0.20        56
           3       0.25      0.26      0.25        58
           4       0.29      0.30      0.29        20
           5       0.87      0.82      0.84       320

    accuracy                           0.62       640
   macro avg       0.43      0.44      0.43       640
weighted avg       0.62      0.62      0.62       640

